# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [ ]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples] # 将字符转换为数字，A->1, B->2, ..., Z->26
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x] # 反转输入序列作为目标输出
    dec_x = [[0]+e_idx[:-1] for e_idx in y] # 解码器输入在目标输出的基础上前面加一个起始符0，去掉最后一个元素
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['GWXGITSKPN', 'ROMNQNMVUS'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 7, 23, 24,  7,  9, 20, 19, 11, 16, 14],
       [18, 15, 13, 14, 17, 14, 13, 22, 21, 19]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 14, 16, 11, 19, 20,  9,  7, 24, 23],
       [ 0, 19, 21, 22, 13, 14, 17, 14, 13, 15]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[14, 16, 11, 19, 20,  9,  7, 24, 23,  7],
       [19, 21, 22, 13, 14, 17, 14, 13, 15, 18]])>)


# 建立sequence to sequence 模型

In [ ]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        完成sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好
        '''
        # 编码
        enc_emb = self.embed_layer(enc_ids) #编码器输入的嵌入表示，shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb) #编码器输出，shape(b_sz, len, h_sz)，编码器最后一个时刻的隐状态，shape(b_sz, h_sz)       
        # 解码
        dec_emb = self.embed_layer(dec_ids) #解码器输入的嵌入表示，shape(b_sz, len, emb_sz)
        dec_out, dec_state = self.decoder(dec_emb, initial_state=enc_state) #解码器输出，shape(b_sz, len, h_sz)，解码器最后一个时刻的隐状态，shape(b_sz, h_sz)       
        # 输出 logits
        logits = self.dense(dec_out) #预测下一个字母
        return logits
    
    
#     @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)，编码器输入
        enc_out, enc_state = self.encoder(enc_emb) # 编码器输出状态
        
        return [enc_out[:, -1, :], enc_state] # 返回编码器最后一个时刻的输出和隐状态
    
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.decoder_cell.call(inp_emb, state) # shape(b_sz, h_sz)，更新状态
        logits = self.dense(h) # shape(b_sz, v_sz)，预测下一个字母
        out = tf.argmax(logits, axis=-1) # 选择概率最高的字母
        return out, state

# Loss函数以及训练逻辑

In [ ]:
@tf.function
def compute_loss(logits, labels): #计算交叉熵损失
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x) #模型前向传播，得到预测的logits
        loss = compute_loss(logits, y) #计算损失

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables) #计算损失函数关于模型可训练变量的梯度
    optimizer.apply_gradients(zip(grads, model.trainable_variables)) #将计算得到的梯度应用于模型的可训练变量，更新模型参数
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000): #训练循环
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [6]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3058236
step 500 : loss 1.5368423
step 1000 : loss 0.86376905
step 1500 : loss 0.6301185
step 2000 : loss 0.42650566
step 2500 : loss 0.35973346


<tf.Tensor: shape=(), dtype=float32, numpy=0.31260318>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [ ]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32) # 解码器的输入初始为全0，shape(b_sz,)，表示起始符
        state = init_state
        collect = []
        for i in range(steps): #逐个生成输出序列的每个元素
            cur_token, state = model.get_next_token(cur_token, state) # 获取下一个预测的token和更新状态
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out # 将数字转换回字符
    
    batched_examples, enc_x, _, _ = get_batch(32, 10) # 生成32个随机字符串
    state = model.encode(enc_x) # 编码输入字符串，得到初始状态
    return decode(state, enc_x.get_shape()[-1]), batched_examples # 返回解码结果和原始输入字符串

def is_reverse(seq, rev_seq): # 判断rev_seq是否是seq的反转字符串
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('WXIJBQOZJG', 'GJZOQBJIXW'), ('GPYPHYPHVX', 'XVHPYHPYPG'), ('BGKZTWXNKA', 'AKNXWTZKGB'), ('XCZFYYSXMZ', 'ZMXSYYFZCX'), ('GOGQQAXXDO', 'ODXXAQQGOG'), ('WUYCPORFLN', 'NLFROPCYUW'), ('FFSRZJNMFV', 'VFMNJZRSFF'), ('GXLLFNWDYA', 'AYDWNFLLXG'), ('HOHWACYTYG', 'GYTYCAWHOH'), ('MKIEKTVZJX', 'XJZVTKEIKM'), ('AMSPHGAUUK', 'KUUAGHPSMA'), ('MYNINISENA', 'ANESININYM'), ('RBATBYPMRA', 'ARMPYBTABR'), ('FPTBCVOLEC', 'CELOVCBTPF'), ('OKMWHJPEKI', 'IKEPJHWMKO'), ('ZMNSYZKBPQ', 'QPBKZYSNMZ'), ('FQEMVPQPPQ', 'QPPQPVMEQF'), ('QAGTBHCJNN', 'IZJVHBVGAQ'), ('ZNCYMDRKJB', 'BJKRDMYCNZ'), ('RGAXZXFDTT', 'TTDFXZXAGR'), ('WCPAXEQAHL', 'LHAQEXAPCW'), ('MGTLBSAOEL', 'LEOASBLTGM'), ('KIWQWZTRHE', 'EHRTZWQWIK'), ('JHSNBOXRSK', 'KSRXOBNSHJ'), ('ZRQEZAHMGG', 'GGMHAZEQRZ'), ('ZHNKTJCUOQ', 'QOUCJTKNHZ'), ('FVJKMDFBVL', 'LVBFDMKJVF